# AMMS 302 — Week 6: Data Warehouse & SQL Queries
**Data warehouse · WHERE · AND/OR/NOT/NULL · AS · ORDER BY · MIN/MAX/AVG/COUNT · DISTINCT · DATE · LIKE · IN · BETWEEN**

> ต่อจาก `healthinfo.db` สัปดาห์ 5 — รันออฟไลน์บน Windows ด้วย `uv run jupyter lab` (ดูวิธีติดตั้งใน [สไลด์ wk05](./wk05.html) & [สไลด์ wk06](./wk06.html))

### 🎯 Learning objectives (CLO1/CLO2)
- แยก OLTP vs Data Warehouse และอธิบาย Star Schema ได้
- กรองด้วย `WHERE` + `AND/OR/NOT` และจัดการ `NULL` (`IS NULL`) ได้
- ใช้ `AS`, `ORDER BY`, `DISTINCT`, `MIN/MAX/AVG/COUNT`, `DATE`, `LIKE`, `IN`, `BETWEEN` ได้

### 📚 Official references (เปิดคู่กับสไลด์)
- SQLite: [docs.html](https://www.sqlite.org/docs.html) · [SELECT](https://www.sqlite.org/lang_select.html) · [Expressions WHERE/LIKE/IN/BETWEEN](https://www.sqlite.org/lang_expr.html) · [Aggregates](https://www.sqlite.org/lang_aggfunc.html) · [Date/Time](https://www.sqlite.org/lang_datefunc.html) · [CLI](https://www.sqlite.org/cli.html)
- Python: [sqlite3](https://docs.python.org/3/library/sqlite3.html) · [tutorial](https://docs.python.org/3/library/sqlite3.html#tutorial)
- Tools: [DB Browser](https://sqlitebrowser.org/) (`scoop install sqlitebrowser`) — สไลด์ 14
- Supplementary: [W3Schools SQL](https://www.w3schools.com/sql/) · [MIMIC-IV hosp](https://mimic.mit.edu/docs/iv/modules/hosp/) · [OMOP CDM](https://ohdsi.github.io/CommonDataModel/)

---

### 🗺️ แผนที่สไลด์ ↔ โน้ตบุ๊ก

| ipynb § | หัวข้อ | สไลด์ wk06 |
|---|---|---|
| §1–2 | WHERE + AND/OR/NOT/NULL | 03–04 |
| §3 | AS alias | 05 |
| §4 | ORDER BY | 06 |
| §5 | Aggregates/DISTINCT | 07 |
| §6 | DATE | 08 |
| §7–9 | LIKE / IN / BETWEEN | 09–11 |
| §10 | Pipeline | 12 |

> เปิดคู่กับ [wk06.html](./wk06.html) — Cheat sheet สไลด์ 15


## 0) เตรียม `healthinfo.db` (จากสัปดาห์ 5)
สัปดาห์ 6 ใช้ไฟล์ `healthinfo.db` เดิม — ถ้าลบไปให้รันเซลล์สร้างใหม่ด้านล่าง (ย่อจาก `week05-sql-basics.ipynb`)


In [ ]:
import sqlite3, pathlib, csv
import pandas as pd
db_path = pathlib.Path("healthinfo.db")
print(f"DB exists: {db_path.exists()} — {db_path.resolve()}")

# ถ้าไม่มี DB ให้สร้างใหม่แบบย่อ (ไม่ต้องรัน week05 ทั้งหมด)
if not db_path.exists():
    print("creating healthinfo.db from patients_data.csv ...")
    con_tmp = sqlite3.connect(db_path)
    cur_tmp = con_tmp.cursor()
    cur_tmp.execute("""CREATE TABLE patients(patient_id INTEGER PRIMARY KEY, hn TEXT UNIQUE NOT NULL, gender TEXT, birth_date TEXT, systolic_bp REAL, hba1c REAL)""")
    # ใส่ 3 แถวตัวอย่าง + โหลด CSV
    cur_tmp.executemany("INSERT OR IGNORE INTO patients VALUES (?,?,?,?,?,?)", [(10001,'HN-54001','ชาย','1960-06-23',129,8.1),(10002,'HN-54002','หญิง','1975-02-10',155,8.3),(10003,'HN-54003','หญิง','1970-01-25',140,None)])
    import csv as _csv
    batch=[]
    with open("patients_data.csv", encoding="utf-8") as f:
        for r in _csv.DictReader(f):
            try: pid=int(float(r["subject_id"]))
            except: continue
            if pid in (10001,10002,10003): continue
            hn=f"HN-{pid}"
            gender=r["gender"] if r["gender"] in ("ชาย","หญิง") else None
            bd=r["admission_date"] if r["admission_date"] and "-" in r["admission_date"] else None
            try: sbp=float(r["systolic_bp"]) if r["systolic_bp"] else None
            except: sbp=None
            try: hba=float(r["HbA1c_level"]) if r["HbA1c_level"] else None
            except: hba=None
            batch.append((pid,hn,gender,bd,sbp,hba))
    cur_tmp.executemany("INSERT OR IGNORE INTO patients VALUES (?,?,?,?,?,?)", batch)
    con_tmp.commit(); con_tmp.close()
    print(f"created with {len(batch)+3} attempted rows")

con = sqlite3.connect(db_path)
cur = con.cursor()
print(f"connected — tables: {cur.execute(\"SELECT name FROM sqlite_master WHERE type='table'\").fetchall()}")
print(f"count: {cur.execute('SELECT COUNT(*) FROM patients').fetchone()[0]}")
print(cur.execute("SELECT sql FROM sqlite_master WHERE name='patients'").fetchone()[0][:200])

## 1) `WHERE` — กรองแถว (สไลด์ 4)
สเปก: [SELECT — WHERE](https://www.sqlite.org/lang_select.html) · [Expressions](https://www.sqlite.org/lang_expr.html) · [W3Schools WHERE](https://www.w3schools.com/sql/sql_where.asp)

ลำดับจริง: `FROM → WHERE → SELECT` — WHERE มาก่อน SELECT (สไลด์ 13)


In [ ]:
print("-- 1.1 หญิงเท่านั้น --")
display(pd.read_sql("SELECT hn, gender FROM patients WHERE gender='หญิง' LIMIT 5", con))
print("count หญิง:", pd.read_sql("SELECT COUNT(*) AS n FROM patients WHERE gender='หญิง'", con).iloc[0,0])

print("\n-- 1.2 HbA1c > 7.0 --")
display(pd.read_sql("SELECT hn, hba1c FROM patients WHERE hba1c > 7.0 LIMIT 5", con))

print("\n-- 1.3 systolic_bp >= 140 --")
display(pd.read_sql("SELECT hn, systolic_bp FROM patients WHERE systolic_bp >= 140 LIMIT 5", con))

print("\n-- 1.4 HN เดียว --")
display(pd.read_sql("SELECT * FROM patients WHERE hn='HN-54001'", con))

## 2) `AND` / `OR` / `NOT` / `NULL` (สไลด์ 5)
สเปก: [IS NULL](https://www.sqlite.org/lang_expr.html#isisnull) · [AND/OR](https://www.w3schools.com/sql/sql_and_or.asp) · [NULL](https://www.w3schools.com/sql/sql_null_values.asp)

⚠️ `= NULL` ได้ 0 แถวเสมอ — ต้อง `IS NULL` — `NOT` ไม่นับแถวที่เป็น NULL


In [ ]:
print("-- 2.1 หญิง AND hba1c > 7 --")
display(pd.read_sql("SELECT hn, gender, hba1c FROM patients WHERE gender='หญิง' AND hba1c > 7.0 LIMIT 5", con))

print("-- 2.2 ชาย OR bp >= 180 --")
display(pd.read_sql("SELECT hn, gender, systolic_bp FROM patients WHERE gender='ชาย' OR systolic_bp >= 180 LIMIT 5", con))

print("-- 2.3 NOT หญิง (ไม่นับ NULL) --")
display(pd.read_sql("SELECT COUNT(*) AS cnt FROM patients WHERE NOT gender='หญิง'", con))
print("NULL gender:", pd.read_sql("SELECT COUNT(*) FROM patients WHERE gender IS NULL", con).iloc[0,0])

print("-- 2.4 IS NULL / IS NOT NULL --")
print(pd.read_sql("SELECT COUNT(*) AS n_null FROM patients WHERE hba1c IS NULL", con))
print(pd.read_sql("SELECT COUNT(*) AS n_not_null FROM patients WHERE hba1c IS NOT NULL", con))
print("-- ผิด: = NULL ได้ 0 แถว --")
print(pd.read_sql("SELECT COUNT(*) AS cnt FROM patients WHERE hba1c = NULL", con))

print("-- 2.5 วงเล็บสำคัญ --")
q1 = pd.read_sql("SELECT COUNT(*) AS n FROM patients WHERE (gender='หญิง' AND hba1c > 7) OR systolic_bp > 180", con).iloc[0,0]
q2 = pd.read_sql("SELECT COUNT(*) AS n FROM patients WHERE gender='หญิง' AND (hba1c > 7 OR systolic_bp > 180)", con).iloc[0,0]
print(f"(หญิง AND a1c>7) OR bp>180: {q1}")
print(f"หญิง AND (a1c>7 OR bp>180): {q2}  — ต่างกัน!")

## 3) `AS` — Alias ตั้งชื่อใหม่ (สไลด์ 6)
สเปก: [SELECT — AS](https://www.sqlite.org/lang_select.html#resultset) · [W3Schools Alias](https://www.w3schools.com/sql/sql_alias.asp)

Alias ใช้ใน `ORDER BY` ได้ แต่ใช้ใน `WHERE` ไม่ได้ (WHERE มาก่อน SELECT)


In [ ]:
print("-- 3.1 คอลัมน์ --")
display(pd.read_sql("SELECT hn AS hospital_no, hba1c AS a1c FROM patients LIMIT 5", con))

print("-- 3.2 นิพจน์ --")
display(pd.read_sql("SELECT systolic_bp AS sbp, hba1c * 10 AS a1c_x10 FROM patients LIMIT 5", con))

print("-- 3.3 ใช้ Alias ใน ORDER BY ได้ --")
display(pd.read_sql("SELECT hn AS hospital_no, hba1c FROM patients WHERE hba1c > 7 ORDER BY hospital_no LIMIT 5", con))

print("-- 3.4 ใช้ Alias ใน WHERE ไม่ได้ (สาธิต error) --")
try:
    pd.read_sql("SELECT hn AS hospital_no FROM patients WHERE hospital_no='HN-54001'", con)
except Exception as e:
    print(f"Error (คาดไว้): {e}")
    print("ต้อง: WHERE hn='HN-54001'")

## 4) `ORDER BY` — เรียงลำดับ (สไลด์ 7)
สเปก: [ORDER BY](https://www.sqlite.org/lang_select.html#orderby) · [W3Schools ORDER BY](https://www.w3schools.com/sql/sql_orderby.asp)

`ASC` (default) น้อย→มาก, `DESC` มาก→น้อย — `LIMIT` โดยไม่มี `ORDER BY` ลำดับสุ่ม


In [ ]:
print("-- 4.1 HbA1c สูงสุด 5 คน --")
display(pd.read_sql("SELECT hn, hba1c FROM patients ORDER BY hba1c DESC LIMIT 5", con))

print("-- 4.2 เรียง 2 ชั้น: gender ASC, hba1c DESC --")
display(pd.read_sql("SELECT gender, hn, hba1c FROM patients ORDER BY gender ASC, hba1c DESC LIMIT 8", con))

print("-- 4.3 วันที่ล่าสุดก่อน (ISO8601) --")
display(pd.read_sql("SELECT hn, birth_date FROM patients ORDER BY birth_date DESC LIMIT 5", con))

print("-- 4.4 NULL อยู่หัว/ท้าย --")
print(pd.read_sql("SELECT hba1c FROM patients ORDER BY hba1c ASC LIMIT 5", con).head())
print("(ASC: NULL มาก่อน) — ลอง ORDER BY hba1c DESC ดู NULL ไปท้าย")

## 5) `MIN` / `MAX` / `AVG` / `COUNT` / `DISTINCT` (สไลด์ 8)
สเปก: [Aggregate Functions](https://www.sqlite.org/lang_aggfunc.html) · [DISTINCT](https://www.sqlite.org/lang_select.html#distinct) · [W3Schools COUNT/AVG](https://www.w3schools.com/sql/sql_count.asp)

`COUNT(*)` นับแถว, `COUNT(col)` ข้าม NULL, `AVG` ข้าม NULL


In [ ]:
print("-- 5.1 สรุปภาพรวม --")
display(pd.read_sql("""
SELECT COUNT(*) AS n_total,
       COUNT(hba1c) AS n_measured,
       AVG(hba1c) AS avg_a1c,
       MIN(hba1c) AS min_a1c,
       MAX(hba1c) AS max_a1c
FROM patients
""", con))

print("-- 5.2 COUNT(DISTINCT) --")
display(pd.read_sql("SELECT COUNT(DISTINCT gender) AS n_genders FROM patients", con))
display(pd.read_sql("SELECT DISTINCT gender FROM patients", con))

print("-- 5.3 AVG ข้าม NULL --")
print(pd.read_sql("SELECT AVG(hba1c) AS avg_ignore_null, AVG(COALESCE(hba1c,0)) AS avg_zero FROM patients", con))

## 6) `DATE` — วันที่ใน SQLite (สไลด์ 9)
สเปก: [Date/Time Functions](https://www.sqlite.org/lang_datefunc.html) · [W3Schools Dates](https://www.w3schools.com/sql/sql_dates.asp)

เก็บเป็น TEXT `YYYY-MM-DD` แล้วใช้ `strftime`, `date`, `julianday`


In [ ]:
print("-- 6.1 ดึงปี/เดือน --")
display(pd.read_sql("SELECT hn, birth_date, strftime('%Y', birth_date) AS y, strftime('%m', birth_date) AS m FROM patients LIMIT 5", con))

print("-- 6.2 อายุ ณ วันนี้ --")
display(pd.read_sql("""
SELECT hn, birth_date,
  CAST((julianday('now') - julianday(birth_date))/365.25 AS INTEGER) AS age
FROM patients LIMIT 5
""", con))

print("-- 6.3 กรองครึ่งปีแรก 2025 (ต้อง ISO8601) --")
display(pd.read_sql("SELECT hn, birth_date FROM patients WHERE birth_date BETWEEN '2025-01-01' AND '2025-06-30' LIMIT 5", con))

print("-- 6.4 ตรวจวันที่ผิดรูปแบบ ('24/05/2025') --")
print(pd.read_sql("SELECT COUNT(*) AS n_slash FROM patients WHERE birth_date LIKE '%/%'", con))
print(pd.read_sql("SELECT COUNT(*) AS n_iso FROM patients WHERE birth_date LIKE '____-__-__'", con))

## 7) `LIKE` — แพทเทิร์น (สไลด์ 10)
สเปก: [LIKE](https://www.sqlite.org/lang_expr.html#like) · [W3Schools LIKE](https://www.w3schools.com/sql/sql_like.asp)

`%` = กี่ตัวก็ได้, `_` = 1 ตัว


In [ ]:
print("-- 7.1 HN ขึ้นต้น HN-54 --")
display(pd.read_sql("SELECT hn FROM patients WHERE hn LIKE 'HN-54%' LIMIT 5", con))

print("-- 7.2 เดือน 02 --")
display(pd.read_sql("SELECT hn, birth_date FROM patients WHERE birth_date LIKE '____-02-__' LIMIT 5", con))

print("-- 7.3 ไม่เอาวันที่ผิดรูปแบบ --")
display(pd.read_sql("SELECT hn, birth_date FROM patients WHERE birth_date NOT LIKE '%/%' LIMIT 5", con))

print("-- 7.4 ลงท้ายด้วย 01 --")
display(pd.read_sql("SELECT hn FROM patients WHERE hn LIKE '%01' LIMIT 5", con))

## 8) `IN` — เทียบหลายค่า (สไลด์ 11)
สเปก: [IN](https://www.sqlite.org/lang_expr.html#in) · [W3Schools IN](https://www.w3schools.com/sql/sql_in.asp)

`IN` แทน `OR` ยาว — `NOT IN` + NULL = กับดัก


In [ ]:
print("-- 8.1 IN --")
display(pd.read_sql("SELECT hn, hba1c FROM patients WHERE hba1c IN (7.0, 7.5, 8.0, 8.1) LIMIT 5", con))
print("-- 8.2 HN กลุ่ม --")
display(pd.read_sql("SELECT * FROM patients WHERE hn IN ('HN-54001','HN-54002','HN-54003')", con))
print("-- 8.3 เทียบ OR vs IN (ผลเท่ากัน แต่ IN สั้นกว่า) --")
n_or = pd.read_sql("SELECT COUNT(*) AS n FROM patients WHERE gender='ชาย' OR gender='หญิง'", con).iloc[0,0]
n_in = pd.read_sql("SELECT COUNT(*) AS n FROM patients WHERE gender IN ('ชาย','หญิง')", con).iloc[0,0]
print(f"OR: {n_or}, IN: {n_in}")

## 9) `BETWEEN` — ช่วงค่า รวมขอบ (สไลด์ 12)
สเปก: [BETWEEN](https://www.sqlite.org/lang_expr.html#between) · [W3Schools BETWEEN](https://www.w3schools.com/sql/sql_between.asp)

`BETWEEN a AND b` = `>= a AND <= b` — รวมขอบ, ใช้ได้ทั้งตัวเลขและวันที่


In [ ]:
print("-- 9.1 HbA1c 6.5–7.0 --")
display(pd.read_sql("SELECT hn, hba1c FROM patients WHERE hba1c BETWEEN 6.5 AND 7.0 LIMIT 5", con))
print("-- 9.2 bp 120–139 --")
display(pd.read_sql("SELECT hn, systolic_bp FROM patients WHERE systolic_bp BETWEEN 120 AND 139 LIMIT 5", con))
print("-- 9.3 วันที่ครึ่งปีแรก --")
display(pd.read_sql("SELECT hn, birth_date FROM patients WHERE birth_date BETWEEN '2025-01-01' AND '2025-06-30' LIMIT 5", con))
print("-- 9.4 NOT BETWEEN --")
display(pd.read_sql("SELECT hn, hba1c FROM patients WHERE hba1c NOT BETWEEN 6.5 AND 7.0 LIMIT 5", con))

## 10) ลำดับการทำงานจริง (สไลด์ 13)
`FROM → WHERE → SELECT → ORDER BY → LIMIT` — เข้าใจแล้วจะไม่สับสน Alias/WHERE


In [ ]:
print("-- 10.1 Alias ใช้ใน ORDER BY ได้ --")
display(pd.read_sql("SELECT hn AS hospital_no, hba1c FROM patients WHERE hba1c > 7 ORDER BY hospital_no LIMIT 5", con))
print("-- 10.2 Alias ใช้ใน WHERE ไม่ได้ --")
try:
    pd.read_sql("SELECT hn AS hospital_no FROM patients WHERE hospital_no='HN-54001'", con)
except Exception as e:
    print(f"Error (คาดไว้): {e}")

print("-- 10.3 WHERE → ORDER BY → LIMIT --")
display(pd.read_sql("SELECT hn, hba1c FROM patients WHERE gender='หญิง' ORDER BY hba1c DESC LIMIT 3", con))

## 11) ปิดการเชื่อมต่อ & Self-check
สไลด์ 15 — เกณฑ์ตรวจ


In [ ]:
con.commit()
print(f"tables: {con.execute(\"SELECT name FROM sqlite_master WHERE type='table'\").fetchall()}")
print(f"total: {con.execute('SELECT COUNT(*) FROM patients').fetchone()[0]}")
print(f"NULL hba1c: {con.execute('SELECT COUNT(*) FROM patients WHERE hba1c IS NULL').fetchone()[0]}")
print(f"DISTINCT gender: {con.execute('SELECT COUNT(DISTINCT gender) FROM patients').fetchone()[0]}")
con.close()
print("closed — เปิดใหม่ด้วย sqlite3.connect('healthinfo.db')")
con2=sqlite3.connect("healthinfo.db")
print(pd.read_sql("SELECT COUNT(*) AS n FROM patients WHERE hba1c BETWEEN 6.5 AND 7", con2))
con2.close()

### 🧾 Cheat Sheet — Week 6 (ตรงกับสไลด์ 15)

| งาน | SQL |
|---|---|
| กรอง | `WHERE x>7 AND (y='a' OR z IS NULL)` |
| เรียง | `ORDER BY a DESC, b ASC` |
| สรุป | `COUNT/AVG/MIN/MAX(col)`, `COUNT(DISTINCT g)` |
| แพทเทิร์น | `LIKE 'HN-54%'` / `'____-02-__'` |
| หลายค่า | `IN (…)` (NOT IN + NULL → 0 แถว!) |
| ช่วง | `BETWEEN 120 AND 139` · `BETWEEN '2025-01-01' AND …` |
| วันที่ | `strftime('%Y',d)` · `julianday('now')` |

**Pipeline:** `FROM → WHERE → SELECT → ORDER BY → LIMIT` — สัปดาห์ 7 เพิ่ม `GROUP BY → HAVING`


### ✅ Self-check (สไลด์ 15)
- `SELECT COUNT(*) FROM patients WHERE hba1c IS NULL;` + `IS NOT NULL` = 103
- `COUNT(DISTINCT gender)` → 2
- `SELECT * FROM patients ORDER BY hba1c DESC LIMIT 3;` ตรงกับ `pd.read_sql`
- อธิบายได้ว่า `WHERE hba1c = NULL` ได้ 0 แถวเพราะต้อง `IS NULL`

### 📝 Homework 6 (ถ้าสั่ง)
ส่ง `.ipynb` — อาจารย์ตรวจด้วย `sqlite3 healthinfo.db "SELECT COUNT(*) FROM patients WHERE hba1c BETWEEN 6.5 AND 7;"`

### 🔜 สัปดาห์หน้า: Week 7 — JOIN / FOREIGN KEY / GROUP BY — ใช้ `healthinfo.db` ต่อ

---
### 🔗 รวมลิงก์สเปก (ซ้ำจากสไลด์ 16)
- [SQLite Docs](https://www.sqlite.org/docs.html) · [SELECT](https://www.sqlite.org/lang_select.html) · [Expressions](https://www.sqlite.org/lang_expr.html) · [Aggregates](https://www.sqlite.org/lang_aggfunc.html) · [Date/Time](https://www.sqlite.org/lang_datefunc.html)
- [Python sqlite3](https://docs.python.org/3/library/sqlite3.html)
- [DB Browser](https://sqlitebrowser.org/) · [W3Schools SQL](https://www.w3schools.com/sql/) — [WHERE](https://www.w3schools.com/sql/sql_where.asp) · [ORDER BY](https://www.w3schools.com/sql/sql_orderby.asp) · [LIKE](https://www.w3schools.com/sql/sql_like.asp) · [IN](https://www.w3schools.com/sql/sql_in.asp) · [BETWEEN](https://www.w3schools.com/sql/sql_between.asp)
- สไลด์: [wk06.html](./wk06.html) — 4(WHERE),5(AND/OR/NULL),6(AS),7(ORDER BY),8(Aggregates),9(DATE),10(LIKE),11(IN),12(BETWEEN),13(Execution order)
